# TweetQA pointer-generator — one-seed experiment

This notebook trains a small encoder–decoder from scratch. At every answer step it mixes
a vocabulary distribution with an attention-based copy distribution over the tweet. It can
therefore copy rare names and generate short words absent from the tweet. It uses train.csv
only; the locked test set is never loaded. Outputs are isolated in a new MyDrive folder.

## Step 1 — Runtime and separate Drive backup

In [17]:
import os,sys,subprocess,shutil,json,csv,zipfile,hashlib
from pathlib import Path
assert "google.colab" in sys.modules
subprocess.run([sys.executable,"-m","pip","install","-q","gensim","emoji","nltk"],check=True)
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime"
print("GPU",torch.cuda.get_device_name(0),"torch",torch.__version__)
WORK=Path("/content/A2_pointer_generator")
RESULTS=WORK/"generator_results"; CKPT=WORK/"generator_checkpoints"
for p in (WORK/"src",WORK/"data",RESULTS,CKPT): p.mkdir(parents=True,exist_ok=True)
os.environ["A2_RESULTS"]=str(RESULTS); os.environ["A2_CKPT"]=str(CKPT)
os.chdir(WORK)
from google.colab import drive
drive.mount("/content/drive")
BACKUP=Path("/content/drive/MyDrive/CITS4012_A2_pointer_generator"); BACKUP.mkdir(parents=True,exist_ok=True)
for name,dst in (("generator_results",RESULTS),("generator_checkpoints",CKPT)):
    if (BACKUP/name).exists(): shutil.copytree(BACKUP/name,dst,dirs_exist_ok=True)
def backup():
    for src,name in ((RESULTS,"generator_results"),(CKPT,"generator_checkpoints")):
        if src.exists(): shutil.copytree(src,BACKUP/name,dirs_exist_ok=True)
    print("Backed up to",BACKUP)


GPU NVIDIA A100-SXM4-80GB torch 2.11.0+cu130
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Upload train.csv

In [18]:
target=WORK/"data/train.csv"
if not target.exists():
    source=next((p for p in [Path("/content/train.csv"),Path("/content/data/train.csv")] if p.exists()),None)
    if source: shutil.copy2(source,target)
    else:
        from google.colab import files
        uploaded=files.upload(); assert "train.csv" in uploaded; target.write_bytes(uploaded["train.csv"])
sha=hashlib.sha1(target.read_bytes().replace(b"\r\n",b"\n")).hexdigest()
assert sha=="8a52bb7dff56b11fdad1ab6fd3b848fbadc22849"
print("DATA READY",target,sha)


DATA READY /content/A2_pointer_generator/data/train.csv 8a52bb7dff56b11fdad1ab6fd3b848fbadc22849


## Step 3 — Embedded source

In [19]:
%%writefile src/__init__.py
# package marker


Writing src/__init__.py


In [20]:
%%writefile src/evaluate.py
"""Evaluation metrics for TweetQA.

Every metric is computed per example as the max over the gold answers,
and corpus scores are the mean x 100.

- BLEU-1: nltk sentence_bleu, weights (1,0,0,0), smoothing method1
- METEOR: nltk meteor_score (needs WordNet)
- ROUGE-L: our own LCS implementation, F-measure with beta = 1.2 (the
  coco-caption convention used by the official TweetQA evaluation)
- EM and token-F1: SQuAD-style, computed on normalised text
"""
import re
import string
import unicodedata
from collections import Counter

METRICS = ("bleu1", "meteor", "rougeL", "em", "f1")
_ARTICLES = re.compile(r"\b(a|an|the)\b")
_nltk_ready = False


def _ensure_nltk():
    global _nltk_ready
    if _nltk_ready:
        return
    import nltk
    for pkg, path in [("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4")]:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)
    _nltk_ready = True


def normalize_answer(s):
    """Lowercase, replace ASCII and Unicode punctuation with spaces,
    drop articles, collapse whitespace."""
    s = s.lower()
    s = "".join(" " if (ch in string.punctuation or unicodedata.category(ch).startswith("P")) else ch
                for ch in s)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


def norm_tokens(s):
    return normalize_answer(s).split()


# ---------------------------------------------------------------- metrics

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred, gold):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    prec, rec = overlap / len(p), overlap / len(g)
    return 2 * prec * rec / (prec + rec)


def lcs_len(a, b):
    """Length of the longest common subsequence of two token lists (O(len(a)*len(b)) DP)."""
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            cur[j] = prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1])
        prev = cur
    return prev[-1]


def rouge_l(pred, gold, beta=1.2):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g and bool(p))
    lcs = lcs_len(p, g)
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(g)
    return (1 + beta ** 2) * prec * rec / (rec + beta ** 2 * prec)


def bleu1(pred, gold):
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return sentence_bleu([g], p, weights=(1, 0, 0, 0),
                         smoothing_function=SmoothingFunction().method1)


def meteor(pred, gold):
    _ensure_nltk()
    from nltk.translate.meteor_score import meteor_score
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return meteor_score([g], p)


_FUNCS = {"bleu1": bleu1, "meteor": meteor, "rougeL": rouge_l, "em": exact_match, "f1": token_f1}


def score_example(pred, golds, metrics=METRICS):
    """Per-example scores in [0, 1]: max over gold answers."""
    pred = pred or ""
    return {m: max(_FUNCS[m](pred, g) for g in golds) for m in metrics}


def evaluate(preds, golds_list, metrics=METRICS, return_per_example=False):
    """preds: list[str]; golds_list: list[list[str]]. Returns corpus scores (mean x 100)."""
    assert len(preds) == len(golds_list), (len(preds), len(golds_list))
    per = [score_example(p, g, metrics) for p, g in zip(preds, golds_list)]
    n = max(1, len(per))
    corpus = {m: 100.0 * sum(s[m] for s in per) / n for m in metrics}
    return (corpus, per) if return_per_example else corpus


def fmt(scores):
    return " ".join(f"{m}={scores[m]:.2f}" for m in METRICS if m in scores)


Writing src/evaluate.py


In [21]:
%%writefile src/data.py
"""TweetQA data pipeline: loading, splits, offset-preserving tokenisation,
tweet normalisation, token flags, silver-span labelling, vocabularies and
a padded DataLoader.

Design notes (see results/DATA_REPORT.md):
- test.csv is the original TweetQA dev set. It stays locked (TEST_LOCKED)
  until the final evaluation. Validation is 10% of train, grouped by tweet.
- Every token keeps its character offsets into the raw tweet, so a
  predicted span (i, j) maps back to tweet[start_i:end_j].
- The normaliser only changes the *word* used for the embedding lookup.
  The char-CNN always sees the raw (lowercased) token, so @handles stay
  distinguishable after @user -> <user>.
"""
import ast
import csv
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset

try:
    from .evaluate import normalize_answer, rouge_l
except ImportError:  # inlined in the notebook / run as a script
    from evaluate import normalize_answer, rouge_l

TEST_LOCKED = True
DATA_DIR = Path(os.environ.get("TWEETQA_DIR", Path(__file__).resolve().parents[1] / "data"
                               if "__file__" in globals() else "data"))
SPLIT_SEED = 4012
VAL_FRAC = 0.10

# ------------------------------------------------------------------ loading

_ANSWER_ITEM = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"", re.S)


def parse_answers(field):
    """The Answer column is a numpy-array repr: ['a' 'b'] (space/newline separated, mixed quotes)."""
    items = [ast.literal_eval(m) for m in _ANSWER_ITEM.findall(field)]
    if not items:
        raise ValueError(f"unparseable answer field: {field!r}")
    return [str(a) for a in items]


def _read_csv(name):
    with open(Path(DATA_DIR) / name, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [dict(row_id=i, qid=r["qid"], question=r["Question"], tweet=r["Tweet"],
                 answers=parse_answers(r["Answer"]), answer_field=r["Answer"])
            for i, r in enumerate(rows)]


def _question_word(q):
    toks = tokenize(q.lower())
    w = toks[0][0] if toks else ""
    return w if w in {"what", "who", "how", "where", "why", "which", "when"} else "other"


def get_splits(seed=SPLIT_SEED, val_frac=VAL_FRAC):
    """Train/val from train.csv: exact duplicates dropped, grouped by tweet,
    stratified by the first question word of each tweet's first question."""
    rows, seen = [], set()
    for r in _read_csv("train.csv"):
        key = (r["question"], r["answer_field"], r["tweet"])
        if key not in seen:
            seen.add(key)
            rows.append(r)
    by_tweet = defaultdict(list)
    for r in rows:
        by_tweet[r["tweet"]].append(r)
    strata = defaultdict(list)
    for tweet in sorted(by_tweet):
        strata[_question_word(by_tweet[tweet][0]["question"])].append(tweet)
    rng = random.Random(seed)
    val_tweets = set()
    for s in sorted(strata):
        tweets = strata[s][:]
        rng.shuffle(tweets)
        val_tweets.update(tweets[:round(len(tweets) * val_frac)])
    train = [r for r in rows if r["tweet"] not in val_tweets]
    val = [r for r in rows if r["tweet"] in val_tweets]
    return train, val


def load_test():
    if TEST_LOCKED:
        raise RuntimeError("Test set is locked (TEST_LOCKED=True). It is used once, in Stage 8.")
    return _read_csv("test.csv")


# ------------------------------------------------------------- tokenisation

_URL = r"https?://\S+|www\.\S+"
_EMOJI_CHARS = "\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍"
_TOKEN_RE = re.compile(
    rf"(?P<url>{_URL})"
    r"|(?P<mention>@\w+)"
    r"|(?P<hashtag>#\w+)"
    rf"|(?P<emoji>[{_EMOJI_CHARS}]+)"
    r"|(?P<number>\d+(?:[.,:/]\d+)*)"
    r"|(?P<word>\w+(?:['’]\w+)*)"
    r"|(?P<sym>[^\w\s])",
    re.UNICODE,
)
_ELONG = re.compile(r"(.)\1{2,}")


def tokenize(text):
    """Returns [(raw_token, start, end, kind)] with char offsets into text."""
    return [(m.group(0), m.start(), m.end(), m.lastgroup) for m in _TOKEN_RE.finditer(text)]


def normalise_tokens(raw_tokens, use_normaliser=True):
    """Map raw tokens to model tokens: dicts with word, raw, start, end, kind.

    Normaliser: @user -> <user>, url -> <url>, emoji -> <emoji>,
    #tag -> '#' + 'tag', elongation squashed to 2 repeats, lowercase.
    With use_normaliser=False the word is just the lowercased raw token.
    """
    out = []
    for raw, s, e, kind in raw_tokens:
        if not use_normaliser:
            out.append(dict(word=raw.lower(), raw=raw, start=s, end=e, kind=kind))
        elif kind == "mention":
            out.append(dict(word="<user>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "url":
            out.append(dict(word="<url>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "emoji":
            out.append(dict(word="<emoji>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "hashtag":
            out.append(dict(word="#", raw="#", start=s, end=s + 1, kind=kind))
            tag = raw[1:]
            out.append(dict(word=_ELONG.sub(r"\1\1", tag.lower()), raw=tag, start=s + 1, end=e, kind=kind))
        else:
            out.append(dict(word=_ELONG.sub(r"\1\1", raw.lower()), raw=raw, start=s, end=e, kind=kind))
    return out


_BYLINE_RE = re.compile(r"\(@\w+\)\s+[A-Za-z]+\.?\s+\d{1,2},\s+\d{4}\s*$")
_BOUNDARY = {".", "!", "?", "—", "–", "-", "\"", "”", "|", ":"}


def byline_start(text, tokens):
    """Index of the first token of the trailing 'Name (@handle) Month D, YYYY'
    byline, or len(tokens) if there is none. The name is taken back to the
    previous sentence boundary, at most 6 tokens."""
    m = _BYLINE_RE.search(text)
    if not m:
        return len(tokens)
    k = next((i for i, t in enumerate(tokens) if t["start"] >= m.start()), len(tokens))
    j = k
    while j > 0 and k - j < 6 and tokens[j - 1]["raw"] not in _BOUNDARY:
        j -= 1
    return j


FLAG_NAMES = ("mention", "hashtag", "url", "emoji", "number", "capitalised", "in_byline", "in_other_seq")


def token_flags(tokens, other_words, byline_from=None):
    by = len(tokens) if byline_from is None else byline_from
    flags = []
    for i, t in enumerate(tokens):
        flags.append([
            float(t["kind"] == "mention"), float(t["kind"] == "hashtag"), float(t["kind"] == "url"),
            float(t["kind"] == "emoji"), float(t["kind"] == "number"),
            float(t["raw"][:1].isupper()), float(i >= by),
            float(t["word"] in other_words and t["word"] not in _STOP),
        ])
    return flags


_STOP = set("""a an the of to in on at for by with from and or but is are was were be been being am
do does did has have had i you he she it we they me him her us them my your his its our their this that
these those what who whom whose which when where why how not no yes so if than then there here as just
about into out up down over can will would should could may might must 's ? . , ! ' " - — : ; ( )""".split())


# ------------------------------------------------------------- silver spans

def _norm_toks(raw):
    return normalize_answer(raw).split()


def silver_spans(tokens, tweet, answers, max_len=15, tau=0.5):
    """Score every span (i, j), j - i < max_len, that starts and ends on a
    content token, by token-F1 against each gold answer (max over golds).

    Returns (best, best_f1, soft):
      best  : (i, j) with the highest F1, ties broken by ROUGE-L then length
      soft  : [(i, j, w)] for all spans with F1 >= tau, weights proportional to F1
    """
    ntok = [_norm_toks(t["raw"]) for t in tokens]
    content = [bool(x) for x in ntok]
    golds = [Counter(_norm_toks(a)) for a in answers]
    glens = [sum(g.values()) for g in golds]
    scored = []
    for i in range(len(tokens)):
        if not content[i]:
            continue
        bag, n = Counter(), 0
        for j in range(i, min(len(tokens), i + max_len)):
            bag.update(ntok[j])
            n += len(ntok[j])
            if not content[j]:
                continue
            f = 0.0
            for g, gl in zip(golds, glens):
                if gl == 0:
                    continue
                ov = sum((bag & g).values())
                if ov:
                    p, r = ov / n, ov / gl
                    f = max(f, 2 * p * r / (p + r))
            if f > 0:
                scored.append((f, i, j))
    if not scored:
        return None, 0.0, []
    top = max(f for f, _, _ in scored)
    ties = [(i, j) for f, i, j in scored if f == top]
    if len(ties) > 1:
        def key(ij):
            txt = tweet[tokens[ij[0]]["start"]:tokens[ij[1]]["end"]]
            return (max(rouge_l(txt, a) for a in answers), -(ij[1] - ij[0]), -ij[0])
        ties.sort(key=key, reverse=True)
    soft = [(i, j, f) for f, i, j in scored if f >= tau]
    z = sum(w for _, _, w in soft)
    soft = [(i, j, w / z) for i, j, w in soft]
    return ties[0], top, soft


def span_text(ex, i, j):
    return ex["tweet"][ex["t_tokens"][i]["start"]:ex["t_tokens"][j]["end"]]


# ------------------------------------------------------------------- vocab

class Vocab:
    PAD, UNK = "<pad>", "<unk>"

    def __init__(self, counter, min_freq=2, specials=()):
        self.itos = [self.PAD, self.UNK] + [s for s in specials if s not in (self.PAD, self.UNK)]
        self.itos += sorted([w for w, c in counter.items() if c >= min_freq and w not in self.itos],
                            key=lambda w: (-counter[w], w))
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def __call__(self, w):
        return self.stoi.get(w, 1)


SPECIALS = ("<user>", "<url>", "<emoji>")
MAX_WORD_CHARS = 16


def build_vocabs(examples, min_freq=2, char_min_freq=5):
    """Vocabularies come from the training split only."""
    wc, cc = Counter(), Counter()
    for ex in examples:
        for t in ex["t_tokens"] + ex["q_tokens"]:
            wc[t["word"]] += 1
            cc.update(t["raw"].lower()[:MAX_WORD_CHARS])
    return Vocab(wc, min_freq, SPECIALS), Vocab(cc, char_min_freq)


# ----------------------------------------------------------------- encoding

# tau: an example is trainable if its best silver span has F1 >= tau.
# soft_tau: minimum F1 for a span to join the soft target set (None = same as tau). Kept separate so the
# soft-set width can be varied without changing which examples are trained on.
DEFAULT_PREP = dict(use_normaliser=True, max_span_len=15, tau=0.5, soft_tau=None)


def preprocess(rows, prep=None):
    """Tokenise, normalise, flag and silver-label raw rows."""
    prep = {**DEFAULT_PREP, **(prep or {})}
    out = []
    for r in rows:
        t_tokens = normalise_tokens(tokenize(r["tweet"]), prep["use_normaliser"])
        q_tokens = normalise_tokens(tokenize(r["question"]), prep["use_normaliser"])
        by = byline_start(r["tweet"], t_tokens)
        t_words = {t["word"] for t in t_tokens}
        q_words = {t["word"] for t in q_tokens}
        soft_tau = prep["tau"] if prep.get("soft_tau") is None else prep["soft_tau"]
        best, best_f1, soft = silver_spans(t_tokens, r["tweet"], r["answers"], prep["max_span_len"], soft_tau)
        out.append(dict(
            r, t_tokens=t_tokens, q_tokens=q_tokens, byline_from=by,
            t_flags=token_flags(t_tokens, q_words, by), q_flags=token_flags(q_tokens, t_words),
            best=best, best_f1=best_f1, soft=soft,
            trainable=best is not None and best_f1 >= prep["tau"],
        ))
    return out


def encode(examples, wvocab, cvocab):
    for ex in examples:
        for side in ("t", "q"):
            toks = ex[f"{side}_tokens"]
            ex[f"{side}_ids"] = [wvocab(t["word"]) for t in toks]
            ex[f"{side}_chars"] = [[cvocab(c) for c in t["raw"].lower()[:MAX_WORD_CHARS]] or [1] for t in toks]
    return examples


class QADataset(Dataset):
    def __init__(self, examples, train=False):
        # during training, only examples with a usable silver span contribute to the loss
        self.examples = [e for e in examples if e["trainable"]] if train else list(examples)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def _pad_side(batch, side, n_flags):
    B = len(batch)
    T = max(1, max(len(ex[f"{side}_ids"]) for ex in batch))
    W = max(1, max((len(c) for ex in batch for c in ex[f"{side}_chars"]), default=1))
    words = torch.zeros(B, T, dtype=torch.long)
    chars = torch.zeros(B, T, W, dtype=torch.long)
    flags = torch.zeros(B, T, n_flags)
    mask = torch.zeros(B, T, dtype=torch.bool)
    for b, ex in enumerate(batch):
        n = len(ex[f"{side}_ids"])
        words[b, :n] = torch.tensor(ex[f"{side}_ids"], dtype=torch.long)
        for k, cs in enumerate(ex[f"{side}_chars"]):
            chars[b, k, :len(cs)] = torch.tensor(cs, dtype=torch.long)
        if n:
            flags[b, :n] = torch.tensor(ex[f"{side}_flags"])
        mask[b, :n] = True
    return words, chars, flags, mask


def collate(batch):
    tw, tc, tf, tm = _pad_side(batch, "t", len(FLAG_NAMES))
    qw, qc, qf, qm = _pad_side(batch, "q", len(FLAG_NAMES))
    B, T = tw.shape
    y_start = torch.full((B,), -1, dtype=torch.long)
    y_end = torch.full((B,), -1, dtype=torch.long)
    soft_start = torch.zeros(B, T)
    soft_end = torch.zeros(B, T)
    for b, ex in enumerate(batch):
        if ex["best"] is not None:
            y_start[b], y_end[b] = ex["best"]
        for i, j, w in ex["soft"]:
            soft_start[b, i] += w
            soft_end[b, j] += w
    return dict(t_words=tw, t_chars=tc, t_flags=tf, t_mask=tm,
                q_words=qw, q_chars=qc, q_flags=qf, q_mask=qm,
                y_start=y_start, y_end=y_end, soft_start=soft_start, soft_end=soft_end,
                meta=batch)


def make_loader(examples, batch_size=32, train=False, seed=0):
    ds = QADataset(examples, train=train)
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=train, collate_fn=collate, generator=g)


# ------------------------------------------------------------------ oracle

def oracle_predictions(examples):
    """Best silver span text for every example ('' if there is no overlapping span)."""
    return [span_text(ex, *ex["best"]) if ex["best"] is not None else "" for ex in examples]


Writing src/data.py


In [22]:
%%writefile src/runlog.py
"""Append-only experiment log: every reported number must come from a row in results/runs.csv."""
import csv
import json
import os
import time
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("A2_RESULTS", Path(__file__).resolve().parents[1] / "results"
                                  if "__file__" in globals() else "results"))
RUNS_CSV = RESULTS_DIR / "runs.csv"
FIELDS = ["run_id", "timestamp", "model", "config", "seed", "split",
          "bleu1", "meteor", "rougeL", "em", "f1", "train_time_s", "notes"]


def log_run(model, config, seed, split, scores, train_time_s=0.0, run_id=None, notes=""):
    RUNS_CSV.parent.mkdir(parents=True, exist_ok=True)
    new = not RUNS_CSV.exists()
    run_id = run_id or f"{model}-{split}-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    row = dict(run_id=run_id, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), model=model,
               config=json.dumps(config, sort_keys=True, default=str), seed=seed, split=split,
               train_time_s=round(train_time_s, 1), notes=notes,
               **{m: round(scores[m], 4) for m in ("bleu1", "meteor", "rougeL", "em", "f1")})
    with open(RUNS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    return run_id


def read_runs():
    if not RUNS_CSV.exists():
        return []
    with open(RUNS_CSV, encoding="utf-8") as f:
        return list(csv.DictReader(f))


Writing src/runlog.py


In [23]:
%%writefile src/model.py
"""TweetBiDAF: an extractive QA model for tweets, trained from scratch.

  1. Embedding: word embedding ⊕ char-CNN over the *raw* token ⊕ token flags
     -> 2-layer highway                                    (A1: use_char_cnn, A2: use_token_flags)
  2. Shared BiLSTM contextual encoder for tweet (H) and question (U)
  3. Trilinear similarity S_ij = w^T [h_i; u_j; h_i ∘ u_j]
     C2Q = softmax_j(S) U,  Q2C = softmax_i(max_j S) H
     G = [h; c2q; h ∘ c2q; h ∘ q2c]                        (A4: biattn_mode "c2q" drops Q2C)
  4. Gated multi-head self-attention over the projected G,
     residual + LayerNorm                                   (A3: use_self_attn)
  5. 2-layer modelling BiLSTM, start head, 1 more BiLSTM layer, end head,
     padding masks, best (i, j) with i <= j < i + L

The design follows BiDAF (Seo et al., 2017) with gated self-attention in the
style of R-Net (Wang et al., 2017). The tweet-specific parts are written for
this project: a char-CNN on raw tokens with a normalised word vocabulary,
token flags (including the byline flag), soft silver-span training, and
self-attention for long-range links between the tweet body and its byline.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

NEG = -1e30

DEFAULT_MODEL = dict(
    word_dim=100, char_dim=32, char_filters=50, char_widths=(3, 4, 5), n_flags=8,
    hidden=128, dropout=0.2, attn_heads=4, max_span_len=15,
    use_char_cnn=True, use_token_flags=True, use_self_attn=True, biattn_mode="both",
    condition_end_on_start=False,
)


class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim, filters, widths):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.convs = nn.ModuleList(nn.Conv1d(char_dim, filters, w, padding=w // 2) for w in widths)
        self.out_dim = filters * len(widths)

    def forward(self, chars):                      # (B, T, W)
        B, T, W = chars.shape
        x = self.emb(chars.view(B * T, W)).transpose(1, 2)            # (BT, c, W)
        cmask = (chars.view(B * T, 1, W) != 0)
        outs = []
        for conv in self.convs:
            y = F.relu(conv(x))[:, :, :W]
            y = y.masked_fill(~cmask, 0.0)          # ReLU output >= 0, so masked max-pool is safe
            outs.append(y.max(dim=2).values)
        return torch.cat(outs, -1).view(B, T, -1)


class Highway(nn.Module):
    def __init__(self, dim, n_layers=2):
        super().__init__()
        self.lin = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))
        self.gate = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))

    def forward(self, x):
        for lin, gate in zip(self.lin, self.gate):
            g = torch.sigmoid(gate(x))
            x = g * F.relu(lin(x)) + (1 - g) * x
        return x


def reverse_padded(x, mask):
    """Reverse each sequence within its own length; padding stays at the end."""
    T = x.size(1)
    lengths = mask.sum(1, keepdim=True)                                  # (B, 1)
    t = torch.arange(T, device=x.device).unsqueeze(0)                    # (1, T)
    idx = torch.where(t < lengths, lengths - 1 - t, t)                   # (B, T)
    return x.gather(1, idx.unsqueeze(-1).expand_as(x))


class BiLSTM(nn.Module):
    """BiLSTM over right-padded batches, padding-exact without packing.

    The forward LSTM reads valid tokens before any padding, so its outputs at
    valid positions are exact. The backward direction runs a forward LSTM over
    each sequence reversed *within its length*, then un-reverses the output.
    The result equals a packed BiLSTM, but runs ~4x faster on MPS.
    """

    def __init__(self, in_dim, hidden, layers=1, dropout=0.0):
        super().__init__()
        self.fw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.bw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        for l, (fw, bw) in enumerate(zip(self.fw, self.bw)):
            if l > 0:
                x = self.drop(x)
            f = fw(x)[0]
            b = reverse_padded(bw(reverse_padded(x, mask))[0], mask)
            x = torch.cat([f, b], -1) * mask.unsqueeze(-1)
        return x


class GatedSelfAttention(nn.Module):
    """Multi-head self-attention, then a sigmoid gate on [x; attended], then residual + LayerNorm."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.gate = nn.Linear(2 * dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        attended, w = self.mha(x, x, x, key_padding_mask=~mask,
                               need_weights=True, average_attn_weights=True)
        g = torch.sigmoid(self.gate(torch.cat([x, attended], -1)))
        return self.norm(x + self.drop(g * attended)), w


class TweetBiDAF(nn.Module):
    def __init__(self, n_words, n_chars, cfg=None, word_vectors=None):
        super().__init__()
        self.cfg = cfg = {**DEFAULT_MODEL, **(cfg or {})}
        H, p = cfg["hidden"], cfg["dropout"]
        self.word_emb = nn.Embedding(n_words, cfg["word_dim"], padding_idx=0)
        if word_vectors is not None:
            self.word_emb.weight.data.copy_(word_vectors)
        emb_dim = cfg["word_dim"]
        if cfg["use_char_cnn"]:
            self.char_cnn = CharCNN(n_chars, cfg["char_dim"], cfg["char_filters"], cfg["char_widths"])
            emb_dim += self.char_cnn.out_dim
        if cfg["use_token_flags"]:
            emb_dim += cfg["n_flags"]
        self.highway = Highway(emb_dim, 2)
        self.drop = nn.Dropout(p)
        self.encoder = BiLSTM(emb_dim, H)
        self.w_sim = nn.Linear(6 * H, 1)           # trilinear: w^T [h; u; h∘u]
        g_dim = 8 * H if cfg["biattn_mode"] == "both" else 6 * H
        self.g_proj = nn.Linear(g_dim, 2 * H)
        if cfg["use_self_attn"]:
            self.self_attn = GatedSelfAttention(2 * H, cfg["attn_heads"], p)
        self.model1 = BiLSTM(2 * H, H, layers=2, dropout=p)
        self.model2 = BiLSTM(2 * H, H)
        self.start_head = nn.Linear(4 * H, 1)
        # Optional boundary-aware head: the predicted start distribution forms a
        # differentiable summary of the likely answer start.  Each end candidate
        # is scored jointly with that summary and their elementwise interaction.
        end_dim = 8 * H if cfg["condition_end_on_start"] else 4 * H
        self.end_head = nn.Linear(end_dim, 1)

    # ---------------------------------------------------------------- layers
    def embed(self, words, chars, flags):
        parts = [self.word_emb(words)]
        if self.cfg["use_char_cnn"]:
            parts.append(self.char_cnn(chars))
        if self.cfg["use_token_flags"]:
            parts.append(flags)
        return self.highway(self.drop(torch.cat(parts, -1)))

    def similarity(self, h, u):
        """S_ij = w_h·h_i + w_u·u_j + w_hu·(h_i∘u_j), computed without materialising [h; u; h∘u]."""
        d = h.size(-1)
        w = self.w_sim.weight.squeeze(0)
        wh, wu, whu = w[:d], w[d:2 * d], w[2 * d:]
        return (h @ wh).unsqueeze(2) + (u @ wu).unsqueeze(1) + (h * whu) @ u.transpose(1, 2) + self.w_sim.bias

    def forward(self, batch, return_attn=False):
        tm, qm = batch["t_mask"], batch["q_mask"]
        h = self.drop(self.encoder(self.embed(batch["t_words"], batch["t_chars"], batch["t_flags"]), tm))
        u = self.drop(self.encoder(self.embed(batch["q_words"], batch["q_chars"], batch["q_flags"]), qm))

        S = self.similarity(h, u)                                        # (B, T, J)
        a = F.softmax(S.masked_fill(~qm.unsqueeze(1), NEG), dim=-1)      # C2Q weights
        c2q = a @ u
        parts = [h, c2q, h * c2q]
        b = None
        if self.cfg["biattn_mode"] == "both":
            smax = S.masked_fill(~qm.unsqueeze(1), NEG).max(-1).values    # (B, T)
            b = F.softmax(smax.masked_fill(~tm, NEG), dim=-1)            # Q2C weights
            q2c = (b.unsqueeze(1) @ h).expand_as(h)
            parts.append(h * q2c)
        x = self.drop(F.relu(self.g_proj(torch.cat(parts, -1))))

        sa = None
        if self.cfg["use_self_attn"]:
            x, sa = self.self_attn(x, tm)

        m1 = self.drop(self.model1(x, tm))
        m2 = self.drop(self.model2(m1, tm))
        start = self.start_head(torch.cat([x, m1], -1)).squeeze(-1).masked_fill(~tm, NEG)
        if self.cfg["condition_end_on_start"]:
            start_prob = F.softmax(start, dim=-1)
            start_summary = torch.bmm(start_prob.unsqueeze(1), m1).expand_as(m2)
            end_features = torch.cat([x, m2, start_summary, m2 * start_summary], -1)
        else:
            start_prob = None
            end_features = torch.cat([x, m2], -1)
        end = self.end_head(end_features).squeeze(-1).masked_fill(~tm, NEG)
        out = dict(start_logits=start, end_logits=end)
        if return_attn:
            out.update(S=S, c2q=a, q2c=b, self_attn=sa,
                       start_prob_for_end=start_prob)
        return out


# ------------------------------------------------------------------ loss / decode

def span_loss(out, batch, soft=True):
    ls = F.log_softmax(out["start_logits"], -1)
    le = F.log_softmax(out["end_logits"], -1)
    if soft:
        # soft cross-entropy against the marginal start/end distributions of the soft silver set
        return -(batch["soft_start"] * ls).sum(-1).mean() - (batch["soft_end"] * le).sum(-1).mean()
    return F.nll_loss(ls, batch["y_start"]) + F.nll_loss(le, batch["y_end"])


@torch.no_grad()
def decode(start_logits, end_logits, max_len=15):
    """argmax over i <= j < i + max_len of log p_s(i) + log p_e(j). Returns (B, 2) indices and scores."""
    ls = F.log_softmax(start_logits, -1)
    le = F.log_softmax(end_logits, -1)
    T = ls.size(1)
    scores = ls.unsqueeze(2) + le.unsqueeze(1)                           # (B, T, T)
    idx = torch.arange(T, device=ls.device)
    band = (idx.unsqueeze(0) >= idx.unsqueeze(1)) & (idx.unsqueeze(0) - idx.unsqueeze(1) < max_len)
    scores = scores.masked_fill(~band, NEG)
    flat = scores.view(scores.size(0), -1)
    best = flat.argmax(-1)
    return torch.stack([best // T, best % T], -1), flat.max(-1).values


Writing src/model.py


In [24]:
%%writefile src/fasttext_init.py
"""Optional in-domain word vectors: FastText (gensim) trained from scratch on the
*training split's* tweets and questions only. No external text and no
pretrained weights, so the main model is still trained from scratch on
TweetQA alone."""
import torch


def fasttext_vectors(wv, cfg, train_examples, epochs=20, seed=1):
    from gensim.models import FastText
    sents, seen = [], set()
    for ex in train_examples:
        if ex["tweet"] not in seen:                 # each tweet once
            seen.add(ex["tweet"])
            sents.append([t["word"] for t in ex["t_tokens"]])
        sents.append([t["word"] for t in ex["q_tokens"]])
    ft = FastText(vector_size=cfg["word_dim"], window=5, min_count=1, sg=1, epochs=epochs,
                  seed=seed, workers=1, min_n=3, max_n=5)
    ft.build_vocab(sents)
    ft.train(sents, total_examples=len(sents), epochs=epochs)
    W = torch.randn(len(wv), cfg["word_dim"]) * 0.1
    W[0] = 0.0
    for i, w in enumerate(wv.itos[2:], start=2):
        W[i] = torch.tensor(ft.wv[w])               # subword n-grams also give OOV-safe vectors
    return W


Writing src/fasttext_init.py


In [25]:
%%writefile src/generative.py
"""Small pointer-generator for TweetQA, trained from scratch on the provided data."""
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

from . import data as D
from .model import BiLSTM, CharCNN, GatedSelfAttention, Highway, NEG

GEN_SPECIALS = ("<pad>", "<unk>", "<bos>", "<eos>")


class GenVocab:
    def __init__(self, tokens, min_freq=2):
        counts = Counter(tokens)
        self.itos = list(GEN_SPECIALS) + sorted(w for w, n in counts.items()
                                                 if n >= min_freq and w not in GEN_SPECIALS)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def id(self, token):
        return self.stoi.get(token, 1)


def raw_answer_tokens(text):
    return [raw.lower() for raw, _, _, _ in D.tokenize(text)]


def build_gen_vocab(train_rows, min_freq=2):
    return GenVocab((t for r in train_rows for a in r["answers"] for t in raw_answer_tokens(a)), min_freq)


def add_generation_fields(examples, gv, max_answer_len=16):
    """Add per-example extended-vocabulary copy ids and decoder targets."""
    V = len(gv)
    for ex in examples:
        source = [t["raw"].lower() for t in ex["t_tokens"]]
        oovs, oov_to_i = [], {}
        copy_ids = []
        for token in source:
            if token in gv.stoi:
                copy_ids.append(gv.stoi[token])
            else:
                if token not in oov_to_i:
                    oov_to_i[token] = len(oovs); oovs.append(token)
                copy_ids.append(V + oov_to_i[token])
        target_tokens = raw_answer_tokens(ex["answers"][0])[:max_answer_len - 1]
        target_ext = [gv.stoi[t] if t in gv.stoi else V + oov_to_i[t] if t in oov_to_i else 1
                      for t in target_tokens] + [3]
        decoder_in = [2] + [i if i < V else 1 for i in target_ext[:-1]]
        ex.update(copy_ids=copy_ids, source_oovs=oovs, target_tokens=target_tokens,
                  target_ext=target_ext, decoder_in=decoder_in)
    return examples


def collate_generation(examples):
    batch = D.collate(examples)
    B, T = batch["t_words"].shape
    L = max(len(e["target_ext"]) for e in examples)
    copy_ids = torch.zeros(B, T, dtype=torch.long)
    decoder_in = torch.zeros(B, L, dtype=torch.long)
    target_ext = torch.zeros(B, L, dtype=torch.long)
    target_mask = torch.zeros(B, L, dtype=torch.bool)
    for b, e in enumerate(examples):
        copy_ids[b, :len(e["copy_ids"])] = torch.tensor(e["copy_ids"])
        n = len(e["target_ext"])
        decoder_in[b, :n] = torch.tensor(e["decoder_in"])
        target_ext[b, :n] = torch.tensor(e["target_ext"])
        target_mask[b, :n] = True
    batch.update(copy_ids=copy_ids, decoder_in=decoder_in, target_ext=target_ext,
                 target_mask=target_mask,
                 max_oov=max((len(e["source_oovs"]) for e in examples), default=0))
    return batch


class TweetPointerGenerator(nn.Module):
    def __init__(self, n_words, n_chars, gen_vocab_size, cfg, word_vectors=None):
        super().__init__()
        self.cfg, self.gen_vocab_size = cfg, gen_vocab_size
        H, p = cfg["hidden"], cfg["dropout"]
        self.word_emb = nn.Embedding(n_words, cfg["word_dim"], padding_idx=0)
        if word_vectors is not None:
            self.word_emb.weight.data.copy_(word_vectors)
        emb_dim = cfg["word_dim"]
        self.char_cnn = CharCNN(n_chars, cfg["char_dim"], cfg["char_filters"], cfg["char_widths"])
        emb_dim += self.char_cnn.out_dim
        if cfg["use_token_flags"]: emb_dim += cfg["n_flags"]
        self.highway = Highway(emb_dim, 2)
        self.drop = nn.Dropout(p)
        self.encoder = BiLSTM(emb_dim, H)
        self.w_sim = nn.Linear(6 * H, 1)
        self.g_proj = nn.Linear(8 * H, 2 * H)
        self.self_attn = GatedSelfAttention(2 * H, cfg["attn_heads"], p)

        dim = 2 * H
        self.dec_emb = nn.Embedding(gen_vocab_size, cfg.get("decoder_emb", 100), padding_idx=0)
        self.attn_mem = nn.Linear(dim, dim, bias=False)
        self.attn_dec = nn.Linear(dim, dim, bias=False)
        self.attn_v = nn.Linear(dim, 1, bias=False)
        self.decoder = nn.GRUCell(cfg.get("decoder_emb", 100) + dim, dim)
        self.vocab_out = nn.Linear(2 * dim, gen_vocab_size)
        self.p_gen = nn.Linear(2 * dim + cfg.get("decoder_emb", 100), 1)

    def embed(self, words, chars, flags):
        parts = [self.word_emb(words), self.char_cnn(chars)]
        if self.cfg["use_token_flags"]: parts.append(flags)
        return self.highway(self.drop(torch.cat(parts, -1)))

    def encode(self, batch):
        tm, qm = batch["t_mask"], batch["q_mask"]
        h = self.drop(self.encoder(self.embed(batch["t_words"], batch["t_chars"], batch["t_flags"]), tm))
        u = self.drop(self.encoder(self.embed(batch["q_words"], batch["q_chars"], batch["q_flags"]), qm))
        d = h.size(-1); w = self.w_sim.weight.squeeze(0)
        S = ((h @ w[:d]).unsqueeze(2) + (u @ w[d:2*d]).unsqueeze(1)
             + (h * w[2*d:]) @ u.transpose(1, 2) + self.w_sim.bias)
        a = F.softmax(S.masked_fill(~qm.unsqueeze(1), NEG), -1); c2q = a @ u
        smax = S.masked_fill(~qm.unsqueeze(1), NEG).max(-1).values
        b = F.softmax(smax.masked_fill(~tm, NEG), -1)
        q2c = (b.unsqueeze(1) @ h).expand_as(h)
        memory = self.drop(F.relu(self.g_proj(torch.cat([h, c2q, h*c2q, h*q2c], -1))))
        memory, _ = self.self_attn(memory, tm)
        return memory

    def attend(self, memory, mask, hidden):
        score = self.attn_v(torch.tanh(self.attn_mem(memory) + self.attn_dec(hidden).unsqueeze(1))).squeeze(-1)
        attn = F.softmax(score.masked_fill(~mask, NEG), -1)
        context = torch.bmm(attn.unsqueeze(1), memory).squeeze(1)
        return context, attn

    def step(self, prev_ids, hidden, memory, mask, copy_ids, max_oov):
        prev_base = prev_ids.masked_fill(prev_ids >= self.gen_vocab_size, 1)
        emb = self.dec_emb(prev_base)
        context, _ = self.attend(memory, mask, hidden)
        hidden = self.decoder(torch.cat([emb, context], -1), hidden)
        context, attn = self.attend(memory, mask, hidden)
        vocab = F.softmax(self.vocab_out(torch.cat([hidden, context], -1)), -1)
        p = torch.sigmoid(self.p_gen(torch.cat([hidden, context, emb], -1)))
        final = vocab.new_zeros(vocab.size(0), self.gen_vocab_size + max_oov)
        final[:, :self.gen_vocab_size] = p * vocab
        final.scatter_add_(1, copy_ids, (1 - p) * attn)
        return final, hidden, attn, p

    def forward(self, batch):
        memory = self.encode(batch); mask = batch["t_mask"]
        hidden = (memory * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        distributions = []
        for t in range(batch["decoder_in"].size(1)):
            dist, hidden, _, _ = self.step(batch["decoder_in"][:, t], hidden, memory, mask,
                                           batch["copy_ids"], batch["max_oov"])
            distributions.append(dist)
        return torch.stack(distributions, 1)

    @torch.no_grad()
    def generate(self, batch, max_len=16):
        memory = self.encode(batch); mask = batch["t_mask"]
        hidden = (memory * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        prev = torch.full((memory.size(0),), 2, dtype=torch.long, device=memory.device)
        outputs, finished = [], torch.zeros(memory.size(0), dtype=torch.bool, device=memory.device)
        for _ in range(max_len):
            dist, hidden, _, _ = self.step(prev, hidden, memory, mask, batch["copy_ids"], batch["max_oov"])
            prev = dist.argmax(-1); outputs.append(prev)
            finished |= prev.eq(3)
            if finished.all(): break
        return torch.stack(outputs, 1)


def generation_loss(distributions, targets, mask):
    prob = distributions.gather(2, targets.unsqueeze(-1)).squeeze(-1).clamp_min(1e-12)
    return -(prob.log() * mask).sum() / mask.sum().clamp_min(1)


Writing src/generative.py


In [26]:
%%writefile src/train_generative.py
"""Train the from-scratch TweetQA pointer-generator on validation only."""
import json, os, random, time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from . import data as D
from .evaluate import evaluate, fmt
from .fasttext_init import fasttext_vectors
from .generative import (TweetPointerGenerator, add_generation_fields, build_gen_vocab,
                         collate_generation, generation_loss)
from .runlog import RESULTS_DIR, log_run


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def move(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def decode_ids(ids, ex, gv):
    out = []
    for i in ids:
        if i == 3: break
        if i in (0, 2): continue
        out.append(gv.itos[i] if i < len(gv) else ex["source_oovs"][i-len(gv)]
                   if i-len(gv) < len(ex["source_oovs"]) else "<unk>")
    return " ".join(out)


@torch.no_grad()
def predict(model, examples, gv, device, batch_size=64, max_len=16):
    model.eval(); out = []
    for start in range(0, len(examples), batch_size):
        chunk = examples[start:start+batch_size]
        batch = move(collate_generation(chunk), device)
        ids = model.generate(batch, max_len).cpu().tolist()
        out.extend(decode_ids(x, e, gv) for x, e in zip(ids, chunk))
    return out


def train(cfg, seed=1, smoke=False):
    seed_all(seed); device = "cuda" if torch.cuda.is_available() else "cpu"
    train_rows, val_rows = D.get_splits()
    if smoke: train_rows, val_rows = train_rows[:200], val_rows[:200]
    prep = dict(use_normaliser=True, max_span_len=cfg["max_span_len"], tau=cfg["tau"])
    tr, va = D.preprocess(train_rows, prep), D.preprocess(val_rows, prep)
    wv, cv = D.build_vocabs(tr, min_freq=cfg["min_freq"])
    D.encode(tr, wv, cv); D.encode(va, wv, cv)
    gv = build_gen_vocab(train_rows, cfg["gen_min_freq"])
    add_generation_fields(tr, gv, cfg["max_answer_len"])
    add_generation_fields(va, gv, cfg["max_answer_len"])
    vectors = fasttext_vectors(wv, cfg, tr, seed=seed)
    model = TweetPointerGenerator(len(wv), len(cv), len(gv), cfg, vectors).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    loader = DataLoader(tr, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=collate_generation,
                        generator=torch.Generator().manual_seed(seed))
    run_id = f"PointerGenerator-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(run_id, "device", device, "params", n_params, "gen_vocab", len(gv), flush=True)
    best = None; bad = 0; history = []; best_preds = []; t0 = time.time()
    epochs = 2 if smoke else cfg["epochs"]
    for epoch in range(1, epochs+1):
        model.train(); total = tokens = 0
        for batch in loader:
            batch = move(batch, device); dist = model(batch)
            loss = generation_loss(dist, batch["target_ext"], batch["target_mask"])
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"]); opt.step()
            total += loss.item() * batch["target_mask"].sum().item(); tokens += batch["target_mask"].sum().item()
        preds = predict(model, va, gv, device, max_len=cfg["max_answer_len"])
        scores = evaluate(preds, [e["answers"] for e in va])
        history.append(dict(epoch=epoch, train_loss=total/max(1,tokens), **scores))
        improved = best is None or scores["rougeL"] > best["rougeL"]
        print(epoch, f"loss={total/max(1,tokens):.4f}", fmt(scores), "*" if improved else "", flush=True)
        if improved:
            best=scores; bad=0; best_preds=preds
            if not smoke:
                ckpt=Path(os.environ["A2_CKPT"]); ckpt.mkdir(parents=True,exist_ok=True)
                torch.save(dict(state_dict=model.state_dict(), cfg=cfg, seed=seed,
                                vocab=wv.itos, chars=cv.itos, gen_vocab=gv.itos), ckpt/f"{run_id}.pt")
        else:
            bad += 1
            if bad >= cfg["patience"]: break
    if smoke:
        assert history[-1]["train_loss"] < history[0]["train_loss"]
        print("GENERATOR SMOKE PASSED"); return
    (RESULTS_DIR/"history").mkdir(parents=True,exist_ok=True)
    (RESULTS_DIR/"preds").mkdir(parents=True,exist_ok=True)
    (RESULTS_DIR/"history"/f"{run_id}.json").write_text(json.dumps(history,indent=1))
    records=[dict(row_id=e["row_id"],qid=e["qid"],pred=p,gold=e["answers"])
             for e,p in zip(va,best_preds)]
    (RESULTS_DIR/"preds"/f"{run_id}.json").write_text(json.dumps(records))
    log_run("PointerGenerator",cfg,seed,"val",best,time.time()-t0,run_id,
            f"params={n_params} device={device}:{torch.cuda.get_device_name(0) if device=='cuda' else ''}")
    print("BEST",fmt(best),run_id)


if __name__ == "__main__":
    cfg=json.loads(Path("generator_config.json").read_text())
    train(cfg, seed=1, smoke="--smoke" in os.sys.argv)


Writing src/train_generative.py


In [27]:
Path('results').mkdir(exist_ok=True)
Path('results/FROZEN_CONFIG.json').write_text('''{
 "_previous": "results/FROZEN_CONFIG_v1.json",
 "_rule": "one-factor sweep (seed 1); variants adopted only if val ROUGE-L > max of 3 base seeds; adopted lr=5e-4 and word_init=fasttext, confirmed with 3 seeds. v2 (2026-10-08, team decision): soft_spans=False, because the A5 ablation (single silver span) beat the soft-target model on all 3 seeds (val ROUGE-L 44.51\u00b10.11 vs 42.95\u00b10.47); ablations re-run on this base in GPU run 2",
 "_run_ids": [
  "A5_single_span-s1-20261007-141322",
  "A5_single_span-s3-20261007-141647",
  "A5_single_span-s2-20261007-141640"
 ],
 "_seeds": [
  1,
  3,
  2
 ],
 "_selected_from": "A5_single_span (Kaggle GPU run 1)",
 "_val_rougeL": [
  44.6186,
  44.5086,
  44.3927
 ],
 "attn_heads": 4,
 "batch_size": 32,
 "biattn_mode": "both",
 "char_dim": 32,
 "char_filters": 50,
 "char_widths": [
  3,
  4,
  5
 ],
 "dropout": 0.2,
 "epochs": 30,
 "grad_clip": 5.0,
 "hidden": 128,
 "lr": 0.0005,
 "max_span_len": 15,
 "min_freq": 2,
 "n_flags": 8,
 "patience": 4,
 "soft_spans": false,
 "tau": 0.5,
 "use_char_cnn": true,
 "use_normaliser": true,
 "use_self_attn": true,
 "use_token_flags": true,
 "weight_decay": 0.0,
 "word_dim": 100,
 "word_init": "fasttext"
}''')

1193

## Step 4 — Generator configuration

In [28]:
frozen=json.loads(Path("results/FROZEN_CONFIG.json").read_text())
cfg={k:v for k,v in frozen.items() if not k.startswith("_")}
cfg.update(epochs=20,patience=4,batch_size=32,lr=0.0005,grad_clip=5.0,
           decoder_emb=100,gen_min_freq=2,max_answer_len=16)
Path("generator_config.json").write_text(json.dumps(cfg,indent=2))
print(json.dumps(cfg,indent=2))


{
  "attn_heads": 4,
  "batch_size": 32,
  "biattn_mode": "both",
  "char_dim": 32,
  "char_filters": 50,
  "char_widths": [
    3,
    4,
    5
  ],
  "dropout": 0.2,
  "epochs": 20,
  "grad_clip": 5.0,
  "hidden": 128,
  "lr": 0.0005,
  "max_span_len": 15,
  "min_freq": 2,
  "n_flags": 8,
  "patience": 4,
  "soft_spans": false,
  "tau": 0.5,
  "use_char_cnn": true,
  "use_normaliser": true,
  "use_self_attn": true,
  "use_token_flags": true,
  "weight_decay": 0.0,
  "word_dim": 100,
  "word_init": "fasttext",
  "decoder_emb": 100,
  "gen_min_freq": 2,
  "max_answer_len": 16
}


## Step 5 — Smoke test and one-seed training

In [29]:
runs=RESULTS/"runs.csv"
done=runs.exists() and any(r["model"]=="PointerGenerator" and r["seed"]=="1"
                           for r in csv.DictReader(runs.open(encoding="utf-8")))
if not done:
    subprocess.run([sys.executable,"-m","src.train_generative","--smoke"],check=True,
                   env={**os.environ,"PYTHONUNBUFFERED":"1"})
    subprocess.run([sys.executable,"-m","src.train_generative"],check=True,
                   env={**os.environ,"PYTHONUNBUFFERED":"1"})
else: print("SKIP completed PointerGenerator seed 1")
backup()


Backed up to /content/drive/MyDrive/CITS4012_A2_pointer_generator


## Step 6 — Extractive/non-extractive analysis

In [30]:
from src import data as D
from src.evaluate import evaluate,token_f1
run=next(r for r in csv.DictReader((RESULTS/"runs.csv").open(encoding="utf-8"))
         if r["model"]=="PointerGenerator" and r["seed"]=="1")
preds=json.loads((RESULTS/"preds"/f"{run['run_id']}.json").read_text())
_,val_rows=D.get_splits(); exs=D.preprocess(val_rows)
by_id={e["row_id"]:e for e in exs}; by_pred={p["row_id"]:p["pred"] for p in preds}
groups={"all":exs,"fully_extractive":[e for e in exs if e["best_f1"]==1.0],
        "partially_or_non_extractive":[e for e in exs if e["best_f1"]<1.0],
        "no_good_span":[e for e in exs if not e["trainable"]]}
summary={"run":run,"slices":{},"examples":[]}
for name,items in groups.items():
    ps=[by_pred[e["row_id"]] for e in items]
    summary["slices"][name]={"n":len(items),**evaluate(ps,[e["answers"] for e in items])}
for e in exs:
    p=by_pred[e["row_id"]]; score=max(token_f1(p,a) for a in e["answers"])
    source={t["raw"].lower() for t in e["t_tokens"]}; pt=[x[0].lower() for x in D.tokenize(p)]
    summary["examples"].append(dict(row_id=e["row_id"],question=e["question"],tweet=e["tweet"],
        gold=e["answers"],prediction=p,f1=round(100*score,2),best_span_f1=round(100*e["best_f1"],2),
        generated_tokens=[t for t in pt if t not in source]))
summary["examples"].sort(key=lambda x:(x["best_span_f1"],-x["f1"]))
(RESULTS/"generator_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps(summary["slices"],indent=2)); backup()


{
  "all": {
    "n": 1076,
    "bleu1": 31.239331582276563,
    "meteor": 25.943900860738406,
    "rougeL": 35.57165386865172,
    "em": 21.189591078066915,
    "f1": 35.76962306977705
  },
  "fully_extractive": {
    "n": 659,
    "bleu1": 45.31450910983712,
    "meteor": 35.91432800191997,
    "rougeL": 49.70236412457615,
    "em": 34.44613050075873,
    "f1": 49.69193169616459
  },
  "partially_or_non_extractive": {
    "n": 417,
    "bleu1": 8.995825609465024,
    "meteor": 10.18727859206059,
    "rougeL": 13.240387540943827,
    "em": 0.23980815347721823,
    "f1": 13.767701283711359
  },
  "no_good_span": {
    "n": 191,
    "bleu1": 1.9027098056564105,
    "meteor": 4.33316643692631,
    "rougeL": 2.8821715889667576,
    "em": 0.5235602094240838,
    "f1": 3.1375440027961385
  }
}
Backed up to /content/drive/MyDrive/CITS4012_A2_pointer_generator


## Step 7 — Package results

In [31]:
archive=Path("/content/pointer_generator_artifacts.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in (RESULTS,CKPT):
        for p in folder.rglob("*"):
            if p.is_file(): z.write(p,p.relative_to(WORK))
shutil.copy2(archive,BACKUP/archive.name)
print("SAVED",BACKUP/archive.name,f"{archive.stat().st_size/1e6:.1f} MB")


SAVED /content/drive/MyDrive/CITS4012_A2_pointer_generator/pointer_generator_artifacts.zip 20.2 MB
